# Modeling — **IndoBERT fine-tuning** (Colab/GPU)

**Tujuan.** Mem-*fine-tune* `indobenchmark/indobert-base-p1` (3 kelas) sebagai
pembanding *deep learning* terhadap *baseline* SVM. Notebook *self-contained* untuk
**Google Colab dengan GPU**; membaca koleksi **`processed_bert`** dari MongoDB Atlas
(kolom `bert`, `label`, lalu `label_id`).

**Sebelum mulai:** menu **Runtime → Change runtime type → GPU (T4)**.

### Beda "fine-tuning" di sini vs tuning SVM

Di SVM yang dituning adalah **hyperparameter** model klasik lewat *grid search*
(`C`, n-gram, `min_df`). Di IndoBERT, *fine-tuning* berarti **melanjutkan pelatihan
bobot jaringan** yang sudah *pretrained* (dilatih pada korpus Indonesia besar) agar
menyesuaikan ke tugas sentimen kita. Jadi yang "dilatih" adalah jutaan parameter
neural network, bukan sekadar memilih kombinasi parameter.

### Hyperparameter fine-tuning yang dipakai

| Hyperparameter | Nilai | Peran |
|----------------|-------|-------|
| `num_train_epochs` | **4** | berapa kali seluruh data train dilewati (lihat catatan di bawah) |
| `learning_rate` | `2e-5` | langkah update bobot — kecil, khas fine-tuning transformer |
| `per_device_train_batch_size` | `16` | jumlah sampel per langkah pelatihan |
| `per_device_eval_batch_size` | `32` | batch saat evaluasi (boleh lebih besar, tanpa gradien) |
| `weight_decay` | `0.01` | regularisasi L2 — tahan *overfitting* |
| `warmup_ratio` | `0.1` | 10% langkah awal LR dinaikkan bertahap → pelatihan lebih stabil |
| `MAX_LEN` | `128` | panjang token maksimum per komentar (sisanya dipangkas) |
| `metric_for_best_model` | `macro_f1` | checkpoint terbaik dipilih dari macro-F1 val |
| `load_best_model_at_end` | `True` | model akhir = checkpoint val terbaik, bukan epoch terakhir |

**Strategi epoch.** Dipakai **4 epoch** + `load_best_model_at_end`. Eksperimen
menaikkan ke 12 epoch + Early Stopping **tidak memperbaiki** test (macro-F1 0,659 →
0,620) — gejala *overfitting*/variansi pada data kecil; maka 4 epoch dipertahankan.
Infrastruktur `EarlyStoppingCallback` tetap ada untuk eksperimen lanjutan.

**Evaluasi** memakai test set yang **identik** dengan SVM, dengan metrik yang sama
(**macro-F1**) agar perbandingan adil.

## 0. Dependency + cek GPU

Pasang `transformers` + `torch` (inti fine-tuning), plus `pymongo`/`certifi` (akses
Atlas), `scikit-learn` (metrik), `matplotlib`, `pandas`/`numpy`. Lalu cek
`torch.cuda.is_available()` — fine-tuning di **CPU sangat lambat**, jadi pastikan
runtime GPU aktif sebelum lanjut.

In [ ]:
%pip install -q "transformers>=4.40" "torch" "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy
import torch
print("CUDA tersedia:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU saja (lambat!)")

## 1. Pilih versi dataset + baca `processed_bert`

Set `VERSION_FLAG` ke salah satu: `in_set6k` (v1), `in_balanced_set` (v2),
`in_set10k` (v3), `in_balanced10k` (v4) — **samakan dengan versi yang diuji di SVM**.
Notebook menarik teks `bert` + `label` dari `processed_bert`, memetakan label ke
`label_id`, lalu **split 70/20/10 yang identik dengan `train_svm.ipynb`**: urut
`comment_id` → potong test 10% → potong val (`0.20/0.90` dari sisa) → train, semua
`stratify` + `seed=42`. Karena prosedur & seed sama persis, **test set-nya identik**
antar model → perbandingan adil. Tempel `MONGO_URI` saat diminta (butuh IP allowlist
`0.0.0.0/0` agar Colab bisa konek).

In [ ]:
import os, pandas as pd
from pymongo import MongoClient
import certifi
from getpass import getpass
from sklearn.model_selection import train_test_split

# === PILIH VERSI DATASET (samakan dgn train_svm utk perbandingan adil) ===
# "in_set6k" (v1 imbalanced 6k) | "in_balanced_set" (v2 balanced 3k)
# "in_set10k" (v3 imbalanced 10k) | "in_balanced10k" (v4 balanced 10k)
# "in_full14k" (v5 full 14k - SEMUA komentar berlabel, imbalanced)
VERSION_FLAG = "in_full14k"
# Suffix file metrik per versi (dipakai saat menyimpan -> dibaca train_svm utk perbandingan).
SUF = {"in_set6k":"is6","in_balanced_set":"ibs","in_set10k":"is10",
       "in_balanced10k":"ib10","in_full14k":"if14"}[VERSION_FLAG]

MONGO_URI = os.environ.get("MONGO_URI","") or getpass("MONGO_URI: ")
DB_NAME = os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")

LABELS=["Negatif","Netral","Positif"]; LABEL2ID={l:i for i,l in enumerate(LABELS)}
df=pd.DataFrame(list(client[DB_NAME]["processed_bert"].find(
    {VERSION_FLAG:True}, {"_id":0,"comment_id":1,"bert":1,"label":1})))
df["label_id"]=df["label"].map(LABEL2ID)
# Split KANONIK: urut comment_id + seed=42 -> identik dengan train_svm utk versi ini.
df=df.sort_values("comment_id").reset_index(drop=True)
tmp,df_test=train_test_split(df,test_size=0.10,stratify=df["label_id"],random_state=42)
df_train,df_val=train_test_split(tmp,test_size=0.20/0.90,stratify=tmp["label_id"],random_state=42)
print(f"VERSI={VERSION_FLAG} | train={len(df_train)} val={len(df_val)} test={len(df_test)}")

## 2. Tokenizer + Dataset

IndoBERT tidak menerima teks mentah — perlu **tokenisasi subword** (WordPiece). Sel ini:

- Muat `AutoTokenizer` milik `indobenchmark/indobert-base-p1` (kosakata harus cocok
  dengan model).
- Potong tiap komentar ke maks **`MAX_LEN = 128` token** (`truncation=True`) dan
  *padding* agar seragam panjang — 128 cukup karena komentar YouTube pendek.
- Bungkus hasil encode + label ke `Dataset` PyTorch (`DS`) untuk train/val/test,
  yang akan disuapkan ke `Trainer`. Memakai field **`bert`** (teks preprocessing
  minimal: lowercase + buang URL/mention/emoji; tanda baca/imbuhan/negasi DIPERTAHANKAN
  agar tokenizer subword bekerja optimal).

In [ ]:
from transformers import AutoTokenizer
import torch

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
SEED = 42
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

ds_train = DS(df_train["bert"].astype(str), df_train["label_id"])
ds_val   = DS(df_val["bert"].astype(str),   df_val["label_id"])
ds_test  = DS(df_test["bert"].astype(str),  df_test["label_id"])
print("Dataset siap.")

## 3. Model + TrainingArguments

Inti konfigurasi fine-tuning:

- **Model**: `AutoModelForSequenceClassification` dari IndoBERT dengan kepala
  klasifikasi **3 kelas** baru (`num_labels=3`) + peta `id2label`/`label2id`.
- **`compute_metrics`**: hitung macro-F1 & accuracy di tiap evaluasi epoch.
- **`TrainingArguments`**: semua hyperparameter (lihat tabel di overview) — LR `2e-5`,
  batch 16, weight decay `0.01`, warmup `0.1`, evaluasi & simpan tiap epoch.
- **Pemilihan model terbaik**: `eval_strategy="epoch"` + `metric_for_best_model="macro_f1"`
  + `load_best_model_at_end=True` → model akhir adalah checkpoint **val terbaik**, bukan
  epoch terakhir (melindungi dari overfit di epoch akhir).
- **`MAX_EPOCHS = 4`** = konfigurasi terbaik pada eksperimen kami; `EarlyStoppingCallback`
  (patience 3) dipertahankan untuk percobaan menaikkan epoch — tapi pada data ini
  menambah epoch tidak membantu (lihat overview).

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id={l: i for i, l in enumerate(LABELS)},
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "accuracy": accuracy_score(p.label_ids, preds)}

# Konfigurasi terpakai: 4 epoch + load_best_model_at_end (pilih checkpoint terbaik di val).
# Catatan eksperimen: menaikkan ke 12 epoch + EarlyStopping JUSTRU menurunkan macro-F1
# test (0.659 -> 0.620) -> overfit/variansi pada data kecil (2.100 train). Maka tetap 4.
# (Boleh naikkan MAX_EPOCHS utk eksperimen; EarlyStopping + best-val checkpoint menjaga.)
MAX_EPOCHS = 4
PATIENCE = 3
args = TrainingArguments(
    output_dir="indobert_out",
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                  eval_dataset=ds_val, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

## 4. Fine-tune

Jalankan `trainer.train()`. Amati log per-epoch: *training loss* turun dan **macro-F1
val** biasanya naik lalu mendatar. Karena `load_best_model_at_end=True`, setelah selesai
`trainer` otomatis memuat ulang bobot checkpoint dengan macro-F1 val tertinggi —
itulah model yang dipakai untuk evaluasi test berikutnya.

In [ ]:
trainer.train()
print("Selesai. Val terbaik:", trainer.evaluate())

## 5. Evaluasi pada test set

Prediksi `ds_test` lalu hitung metrik dengan **fungsi `evaluate` yang setara dengan
notebook SVM**: accuracy, macro-F1 (metrik utama), weighted-F1, precision/recall/F1
per kelas, dan confusion matrix. Karena test set identik dengan SVM, angka macro-F1 di
sini bisa langsung diadu dengan baseline SVM secara *apple-to-apple*.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=1).tolist()
y_true = df_test["label_id"].tolist()

def evaluate(y_true, y_pred, labels=LABELS):
    ids = list(range(len(labels)))
    rep = classification_report(y_true, y_pred, labels=ids, target_names=labels, output_dict=True, zero_division=0)
    return {"accuracy": round(accuracy_score(y_true,y_pred),4),
            "macro_f1": round(f1_score(y_true,y_pred,average="macro",zero_division=0),4),
            "weighted_f1": round(f1_score(y_true,y_pred,average="weighted",zero_division=0),4),
            "per_class": {l:{k:round(rep[l][k],4) for k in ["precision","recall","f1-score"]} | {"support":int(rep[l]["support"])} for l in labels},
            "confusion_matrix": confusion_matrix(y_true,y_pred,labels=ids).tolist(), "labels": labels}

m_test = evaluate(y_true, y_pred)
print("="*60); print("  IndoBERT — TEST"); print("="*60)
print(f"  Accuracy : {m_test['accuracy']:.4f}")
print(f"  Macro-F1 : {m_test['macro_f1']:.4f}   <-- metrik utama")
for l in LABELS:
    pc = m_test["per_class"][l]
    print(f"    {l:<10} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1-score']:.3f}")

## 6. Confusion matrix + simpan metrik

Gambar confusion matrix test (lihat kelas mana yang paling sering tertukar — biasanya
**Netral**) lalu simpan dua artefak: `indobert_metrics.json` (metrik lengkap) dan
`indobert_test_confusion.png`. **Penting:** unduh `indobert_metrics.json` dan letakkan
di `outputs/reports/` repo lokal dengan nama bersuffix versi (`indobert_metrics_ibs.json`
dst.) agar sel perbandingan akhir di `train_svm.ipynb` bisa memuatnya.

In [ ]:
import matplotlib.pyplot as plt, json
cm = np.array(m_test["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5,4.3))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title("IndoBERT — Test")
th = cm.max()/2
for i in range(3):
    for j in range(3):
        ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>th else "black")
fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout(); plt.show()

_mfile = f"indobert_metrics_{SUF}.json"; _cfile = f"indobert_test_confusion_{SUF}.png"
json.dump({"model":"IndoBERT","version":VERSION_FLAG,"test":m_test}, open(_mfile,"w"), ensure_ascii=False, indent=2)
fig.savefig(_cfile, dpi=120)
print(f"Tersimpan: {_mfile} + {_cfile}")
print(f"-> taruh {_mfile} di outputs/reports/ agar dibaca train_svm_full14k (perbandingan v5).")
# (Opsional) unduh ke lokal:
# from google.colab import files
# files.download(_mfile); files.download(_cfile)

**Bandingkan dengan SVM.** *Baseline* SVM: **macro-F1 = 0,694** (test).

Setelah notebook ini selesai, salin `indobert_metrics.json` ke
`outputs/reports/` lalu jalankan ulang **sel #9 di `train_svm.ipynb`** untuk
memunculkan tabel + grafik perbandingan akhir SVM vs IndoBERT.